# Hệ Thống Hybrid Quantum-Classical QML 🚀

**Mục tiêu:** Sử dụng QCNN (Pennylane + PyTorch) làm Feature Extractor để trích xuất không gian lượng tử đa chiều, sau đó kết hợp với các đặc trưng cổ điển để huấn luyện XGBoost và CatBoost.

**Đặc trưng Lượng tử:**
- Chỉ sử dụng 8 Features Mới (sinh ra từ Feature Engineering) nạp vào hệ thống lượng tử.
- Đầu ra lượng tử có thể là 8 Expectation Values hoặc 256 Statevector Probabilities.


In [1]:
!pip install -q pennylane torch xgboost catboost seaborn matplotlib-venn openpyxl upsetplot
import pandas as pd
import numpy as np
import time
import os
import gc
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, precision_recall_curve

import torch
import torch.nn as nn
import pennylane as qml
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib_venn import venn2, venn3
from upsetplot import plot as upset_plot, from_memberships
import xgboost as xgb
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings("ignore")

# ==========================================
# CẤU HÌNH HỆ THỐNG HYBRID QML SOTA
# ==========================================
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# --- 1. CẤU HÌNH FEATURE LƯỢNG TỬ ---
QUANTUM_FEATURE_TYPE = 'EXPECTATION'
# 'EXPECTATION': Đo lường giá trị kỳ vọng Z tại từng Qubit (8 features).
# 'STATEVECTOR': Trích xuất toàn bộ véc-tơ phân phối xác suất (256 features).

# --- 2. CẤU HÌNH KIẾN TRÚC QCNN (FIXED) ---
CONV_DEPTHS = [6, 6, 5]  

# --- 3. CẤU HÌNH CLASSICAL HEADS (SOTA PARAMS) ---
# Dựa trên best params từ attacker-2026 - classic.ipynb
# 🌲 XGBoost SOTA
XGB_N_ESTIMATORS = 1000        # Số lượng cây tối đa (Dùng kết hợp Early Stopping)
XGB_LEARNING_RATE = 0.03       # Tốc độ học (Nhỏ giúp mô hình hội tụ từ từ, chắc chắn)
XGB_MAX_DEPTH = 5              # Độ sâu tối đa (5 để tránh overfit quá mức)
XGB_SUBSAMPLE = 0.8            # Tỷ lệ lấy mẫu dữ liệu ngẫu nhiên cho mỗi cây (80%)
XGB_COLSAMPLE_BYTREE = 0.8     # Tỷ lệ lấy mẫu feature ngẫu nhiên cho mỗi cây (80%)
XGB_REG_ALPHA = 0.5            # Phạt L1 (Lasso) để loại bỏ feature lượng tử rác
XGB_REG_LAMBDA = 2.0           # Phạt L2 (Ridge) để chống bùng nổ trọng số
XGB_EARLY_STOPPING = 50        # Dừng sớm nếu sau 50 cây không giảm loss trên tập Val

# 🐱 CatBoost SOTA
CB_ITERATIONS = 1000
CB_LEARNING_RATE = 0.03
CB_DEPTH = 6
CB_L2_LEAF_REG = 5.0           # Phạt L2 cực mạnh đặc trưng của CatBoost
CB_EARLY_STOPPING = 50

os.makedirs('hybrid_results', exist_ok=True)
print("✅ Khởi tạo Cấu hình Hybrid SOTA thành công!")


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Khởi tạo Cấu hình Hybrid SOTA thành công!


## 1. Cấu hình Hệ thống (Config)


In [2]:
import pandas as pd
import numpy as np
import time
import os
import gc
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, precision_recall_curve

import torch
import torch.nn as nn
import pennylane as qml
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib_venn import venn2, venn3
import xgboost as xgb
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings("ignore")

# ==========================================
# CẤU HÌNH HỆ THỐNG HYBRID QML
# ==========================================
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# --- 1. CẤU HÌNH FEATURE LƯỢNG TỬ ---
QUANTUM_FEATURE_TYPE = 'EXPECTATION'
# 'EXPECTATION': Đo lường giá trị kỳ vọng Z tại từng Qubit trước Pooling cuối (Tạo 8 features).
# 'STATEVECTOR': Trích xuất toàn bộ véc-tơ phân phối xác suất (Tạo 256 features).

# --- 2. CẤU HÌNH KIẾN TRÚC QCNN (FIXED) ---
CONV_DEPTHS = [6, 6, 5]  
# Mạch cố định, dùng làm hàm băm phi tuyến chiếu dữ liệu.

# --- 3. CẤU HÌNH HUẤN LUYỆN (DYNAMIC LR) ---
MAX_EPOCHS = 100
BATCH_SIZE = 64
LEARNING_RATE = 0.01

AUTO_REDUCE_LR = True
LR_PATIENCE = 3
LR_FACTOR = 0.5
EARLY_STOPPING_PATIENCE = 5

os.makedirs('hybrid_results', exist_ok=True)
print("✅ Khởi tạo Cấu hình Hybrid thành công!")
print(f"   - Phương pháp trích xuất lượng tử: {QUANTUM_FEATURE_TYPE}")
print(f"   - Độ sâu mạch (Fixed): {CONV_DEPTHS}")
# --- 3. CẤU HÌNH CLASSICAL HEADS & FUSION ---
CLASSIC_FEATURE_MODE = 'QUANTUM_ONLY' 
# Chọn 1 trong 5: 'QUANTUM_ONLY', 'QUANTUM_ALL', 'QUANTUM_ONLY_NEW', 'QUANTUM_ONLY_ORIGINAL', 'QUANTUM_NEW_AND_UNUSED_ORIGINAL'

# Tham số XGBoost (Gợi ý: max_depth thấp để tránh overfit feature lượng tử)
XGB_LEARNING_RATE = 0.05
XGB_MAX_DEPTH = 6
XGB_SUBSAMPLE = 0.8
XGB_COLSAMPLE_BYTREE = 0.8

# Tham số CatBoost (Gợi ý: L2 mạnh để chặn bùng nổ trọng số lá)
CB_LEARNING_RATE = 0.05
CB_DEPTH = 6
CB_L2_LEAF_REG = 3


✅ Khởi tạo Cấu hình Hybrid thành công!
   - Phương pháp trích xuất lượng tử: EXPECTATION
   - Độ sâu mạch (Fixed): [6, 6, 5]


## 2. Tiền xử lý dữ liệu (Classic Feature Engineering)


In [3]:
# Đọc dữ liệu
df = pd.read_excel('/kaggle/input/attacker-2026-credit-risk/Credit Risk Dataset (1).xlsx')
X = df.drop(columns=['loan_status'])
y = df['loan_status']

numeric_features = ['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
                    'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']
categorical_features = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']

# --- Chia dữ liệu (Ratio) ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)
print(f"Dữ liệu gốc Train: {X_train.shape} | Test: {X_test.shape}")

# --- Feature Engineering ---
def engineer_features(X_tr, X_te):
    X_tr, X_te = X_tr.copy(), X_te.copy()
    for s in [X_tr, X_te]:
        s['income_per_age']             = s['person_income'] / s['person_age']
        s['emp_stability_ratio']        = s['person_emp_length'] / (s['person_age'] - 18 + 1e-5)
        s['loan_to_emp_length']         = s['loan_amnt'] / (s['person_emp_length'] + 1)
        s['estimated_interest_burden']  = s['loan_amnt'] * (s['loan_int_rate'] / 100)
        s['is_anomaly_emp']             = (s['person_emp_length'] > (s['person_age'] - 14)).astype(int)
        s['is_extreme_loan']            = (s['loan_percent_income'] > 0.5).astype(int)
    h = X_tr.groupby('person_home_ownership')['person_income'].mean()
    g = X_tr.groupby('loan_grade')['loan_amnt'].mean()
    for s in [X_tr, X_te]:
        s['income_vs_home_mean'] = s['person_income'] / s['person_home_ownership'].map(h)
        s['loan_vs_grade_mean']  = s['loan_amnt']    / s['loan_grade'].map(g)
    X_te[['income_vs_home_mean','loan_vs_grade_mean']] = \
        X_te[['income_vs_home_mean','loan_vs_grade_mean']].fillna(1.0)
    return X_tr, X_te

X_train_eng, X_test_eng = engineer_features(X_train, X_test)

# 8 Features mới
new_num_cols = ['income_per_age','emp_stability_ratio','loan_to_emp_length',
                'estimated_interest_burden','is_anomaly_emp','is_extreme_loan',
                'income_vs_home_mean','loan_vs_grade_mean']

print(f"✅ Tạo thành công 8 Đặc trưng mới!")

Dữ liệu gốc Train: (26064, 28) | Test: (6517, 28)
✅ Tạo thành công 8 Đặc trưng mới!


## 3. Tiền xử lý Dữ liệu Lượng Tử (Tái tạo PCA 8D khớp với Model cũ)


In [4]:
# =====================================================================
# TÁI TẠO BỘ TIỀN XỬ LÝ PCA Y HỆT FILE GỐC ĐỂ DÙNG LẠI MODEL CŨ
# (Sử dụng cấu hình ONLY_NEW + PCA 8D)
# =====================================================================
from sklearn.decomposition import PCA

# 1. Trích xuất đúng 8 Features mới (ONLY_NEW)
X_train_q = X_train_eng[new_num_cols].copy()
X_test_q  = X_test_eng[new_num_cols].copy()

# 2. Chuẩn hóa (StandardScaler)
q_preprocessor = Pipeline([
    ('imp', SimpleImputer(strategy='median')), 
    ('sc', StandardScaler())
])

X_train_q_std = q_preprocessor.fit_transform(X_train_q).astype('float32')
X_test_q_std = q_preprocessor.transform(X_test_q).astype('float32')

# 3. Ép PCA 8D (Thực chất là phép xoay trục và giải tương quan)
pca = PCA(n_components=8, svd_solver='full')
X_train_pca = pca.fit_transform(X_train_q_std).astype('float32')
X_test_pca = pca.transform(X_test_q_std).astype('float32')

ev = np.sum(pca.explained_variance_ratio_)
print(f"📊 PCA: 8D → 8D | Phép quay trục (Rotation) giữ lại {ev*100:.2f}% thông tin")

# 4. Map về miền [-π, π] cho Angle Embedding
A_train = (np.tanh(X_train_pca) * np.pi).astype('float32')
A_test  = (np.tanh(X_test_pca) * np.pi).astype('float32')

n_qubits = 8

print(f"✅ Đã chuẩn bị dữ liệu cho Lượng Tử: Map vào {n_qubits} Qubits bằng PCA (Từ ONLY_NEW)!")


📊 PCA: 8D → 8D | Phép quay trục (Rotation) giữ lại 100.00% thông tin
✅ Đã chuẩn bị dữ liệu cho Lượng Tử: Map vào 8 Qubits bằng PCA (Từ ONLY_NEW)!


## 4. Kiến trúc Mạch Lượng Tử (Pennylane + PyTorch)


In [5]:
# Hàm tính tổng tham số
def compute_qcnn_param_count(n_qubits, conv_depths):
    total, wires = 0, list(range(n_qubits))
    layer = 1
    while len(wires) > 1:
        n_pairs = len(wires) // 2
        depth = conv_depths[layer - 1] if layer - 1 < len(conv_depths) else 1
        conv_params = n_pairs * 3 * depth
        pool_params = n_pairs
        total += conv_params + pool_params
        
        kept = [wires[2*j+1] for j in range(n_pairs)]
        if len(wires) % 2 == 1:
            kept.append(wires[-1])
        wires = kept
        layer += 1
    return total, wires[0]

# Xây dựng mạch Pennylane
def build_qcnn(n_qubits, conv_depths):
    dev = qml.device('default.qubit', wires=n_qubits)

    wire_layers = []
    wires = list(range(n_qubits))
    while len(wires) > 1:
        wire_layers.append(list(wires))
        n_pairs = len(wires) // 2
        kept = [wires[2*j+1] for j in range(n_pairs)]
        if len(wires) % 2 == 1:
            kept.append(wires[-1])
        wires = kept
    final_wire = wires[0]
    
    def qcnn_ansatz(inputs, params):
        # 1. Angle Embedding
        qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation='Y')
        
        # 2. Conv & Pool
        p = 0
        layer = 1
        for layer_wires in wire_layers:
            n_pairs = len(layer_wires) // 2
            depth = conv_depths[layer - 1] if layer - 1 < len(conv_depths) else 1
            for d in range(depth):
                for j in range(n_pairs):
                    w1, w2 = layer_wires[2*j], layer_wires[2*j+1]
                    qml.RY(params[p],   wires=w1)
                    qml.RY(params[p+1], wires=w2)
                    qml.CNOT(wires=[w1, w2])
                    qml.RZ(params[p+2], wires=w1)
                    p += 3
            for j in range(n_pairs):
                w1, w2 = layer_wires[2*j], layer_wires[2*j+1]
                qml.CRZ(params[p], wires=[w1, w2])
                p += 1
            layer += 1

    # Mạch dùng để TRAIN (phân loại)
    @qml.qnode(dev, interface='torch', diff_method='best')
    def train_circuit(inputs, params):
        qcnn_ansatz(inputs, params)
        return qml.probs(wires=[final_wire])

    # Mạch dùng để FEATURE EXTRACTION (Trích xuất)
    @qml.qnode(dev, interface='torch', diff_method='best')
    def feature_circuit(inputs, params):
        qcnn_ansatz(inputs, params)
        if QUANTUM_FEATURE_TYPE == 'EXPECTATION':
            # 8 Features (Z-expectation tại tất cả qubit)
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
        else:
            # 256 Features (Statevector probabilities)
            return qml.probs(wires=range(n_qubits))

    n_params, _ = compute_qcnn_param_count(n_qubits, conv_depths)
    return train_circuit, feature_circuit, n_params

class HybridQCNN(nn.Module):
    def __init__(self, n_qubits, conv_depths):
        super().__init__()
        self.train_circuit, self.feature_circuit, n_params = build_qcnn(n_qubits, conv_depths)
        self.circuit_params = nn.Parameter(0.05 * torch.randn(n_params))
        self.head = nn.Sequential(
            nn.Linear(2, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        )

    def forward(self, x):
        # Trả ra output 2 chiều để tối ưu CrossEntropyLoss
        q_out = self.train_circuit(x, self.circuit_params)
        return self.head(q_out.to(torch.float32))

    def extract_features(self, x):
        # Trả ra N-dimensional tensor cho mô hình Classical
        with torch.no_grad():
            features = self.feature_circuit(x, self.circuit_params)
            if QUANTUM_FEATURE_TYPE == 'EXPECTATION':
                # features là list của 8 tensors, ta stack lại
                features = torch.stack(features, dim=-1)
            return features.numpy()
            
print("✅ Định nghĩa Kiến trúc QCNN (Angle-NoPCA) hoàn tất!")

✅ Định nghĩa Kiến trúc QCNN (Angle-NoPCA) hoàn tất!


## 5. Nạp mô hình QCNN (Bỏ qua Huấn luyện)


In [6]:
print(f"\n{'='*60}\n🚀 NẠP TRỌNG SỐ QCNN TỪ FILE (Bỏ qua Huấn luyện)\n{'='*60}")

model = HybridQCNN(n_qubits, CONV_DEPTHS)

# Thay đường dẫn này bằng đường dẫn tới file model cũ của bạn trên Kaggle
MODEL_PATH = '/kaggle/input/attacker-2026-credit-risk/best_model.pth' 

if os.path.exists(MODEL_PATH):
    # Nạp trọng số
    checkpoint = torch.load(MODEL_PATH, map_location='cpu', weights_only=True)
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
    print(f"✅ Đã nạp thành công trọng số QCNN từ {MODEL_PATH}!")
    
    # -------------------------------------------------------------
    # BƯỚC XÁC MINH (SANITY CHECK) TRÊN TẬP TEST
    # -------------------------------------------------------------
    print("\n🔍 Đang chạy kiểm tra (Sanity Check) model độc lập trên tập Test...")
    model.eval()
    with torch.no_grad():
        probs_test = []
        for i in range(0, len(A_test), 256):
            probs_test.append(torch.softmax(model.train_circuit(torch.tensor(A_test[i:i+256]), model.circuit_params), 1)[:, 1])
        test_probs = torch.cat(probs_test).numpy()
        
        # Vì model gốc được nối thêm một head PyTorch nhỏ (Linear -> ReLU -> Linear) để xuất ra 2 class
        # Ta gọi hàm forward đầy đủ của model để ra output xác suất thực sự.
        probs_test_full = []
        for i in range(0, len(A_test), 256):
            probs_test_full.append(torch.softmax(model(torch.tensor(A_test[i:i+256])), 1)[:, 1])
        test_probs_full = torch.cat(probs_test_full).numpy()
        
        val_auc = roc_auc_score(y_test, test_probs_full)
        print(f"🎯 Kết quả Đánh giá QCNN Standalone trên Test Set:")
        print(f"   ROC AUC = {val_auc:.4f}")
        print("   (Lưu ý: Nếu điểm số này khớp với AUC trong notebook cũ của bạn, quá trình nạp dữ liệu và trọng số đã thành công mỹ mãn!)")
        
else:
    print(f"⚠️ CẢNH BÁO: Không tìm thấy tệp {MODEL_PATH}.")
    print("Vui lòng tải tệp .pth lên Kaggle và sửa lại đường dẫn ở biến MODEL_PATH phía trên.")



🚀 NẠP TRỌNG SỐ QCNN TỪ FILE (Bỏ qua Huấn luyện)
✅ Đã nạp thành công trọng số QCNN từ /kaggle/input/attacker-2026-credit-risk/best_model.pth!

🔍 Đang chạy kiểm tra (Sanity Check) model độc lập trên tập Test...
🎯 Kết quả Đánh giá QCNN Standalone trên Test Set:
   ROC AUC = 0.8219
   (Lưu ý: Nếu điểm số này khớp với AUC trong notebook cũ của bạn, quá trình nạp dữ liệu và trọng số đã thành công mỹ mãn!)


## 6. Trích xuất Đặc trưng Lượng tử (Quantum Features)


In [7]:
print(f"👉 Bắt đầu trích xuất Quantum Features (Chế độ: {QUANTUM_FEATURE_TYPE})...")
model.eval()

# Trích xuất dạng batch để không tràn RAM với Statevector
def extract_in_batches(model, X_arr, batch_size=256):
    all_feats = []
    for i in range(0, len(X_arr), batch_size):
        batch = torch.tensor(X_arr[i:i+batch_size])
        feats = model.extract_features(batch)
        all_feats.append(feats)
    return np.vstack(all_feats)

Q_Train = extract_in_batches(model, A_train)
Q_Test  = extract_in_batches(model, A_test)

print(f"✅ Trích xuất thành công! Shape Train: {Q_Train.shape} | Shape Test: {Q_Test.shape}")

👉 Bắt đầu trích xuất Quantum Features (Chế độ: EXPECTATION)...
✅ Trích xuất thành công! Shape Train: (26064, 8) | Shape Test: (6517, 8)


## 7. Hợp nhất Dữ liệu (Data Fusion - Hybrid Dataset)


## SIÊU DUNG HỢP (HYPER FUSION) - QUANTUM_ONLY_ORIGINAL
Trộn 8 tính năng Lượng tử vào 7 tính năng Cổ điển gốc và 4 biến Phân loại. **Tuyệt đối không dùng các biến xào nấu (Feature Engineering).**

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
import os
os.makedirs('saved_models', exist_ok=True)

original_num = ['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
                'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']
cat_cols = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']
q_cols = [f'q_{i}' for i in range(8)]

numeric_features = q_cols + original_num
categorical_features = cat_cols

data_dict = {}
print("🔄 Đang thiết lập các phiên bản dữ liệu đưa vào 3 Pipelines (Hyper Fusion)...")

# --- LUỒNG 1: STANDARD (OHE + SCALED + SMOTE) ---
preprocessor_standard = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), original_num),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
])

C_Tr_Std = preprocessor_standard.fit_transform(X_train_eng)
C_Te_Std = preprocessor_standard.transform(X_test_eng)

X_train_std = np.hstack((Q_Train, C_Tr_Std)).astype(np.float32)
X_test_std = np.hstack((Q_Test, C_Te_Std)).astype(np.float32)

smote = SMOTE(random_state=RANDOM_SEED)
X_train_smote, y_train_smote = smote.fit_resample(X_train_std, y_train)

data_dict['standard'] = {
    'X_train': X_train_smote, 'y_train': y_train_smote, 
    'X_test': X_test_std, 'y_test': y_test
}

# --- LUỒNG 2: NATIVE CATEGORICAL ---
preprocessor_native = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), original_num),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), cat_cols)
])

C_Tr_Nat = preprocessor_native.fit_transform(X_train_eng)
C_Te_Nat = preprocessor_native.transform(X_test_eng)

X_train_nat = np.hstack((Q_Train, C_Tr_Nat))
X_test_nat = np.hstack((Q_Test, C_Te_Nat))

USE_SMOTE_FOR_TREES = False
if USE_SMOTE_FOR_TREES:
    X_train_tree, y_train_tree = smote.fit_resample(X_train_nat, y_train)
else:
    X_train_tree, y_train_tree = X_train_nat, y_train.values

data_dict['native'] = {
    'X_train': X_train_tree, 'y_train': y_train_tree, 
    'X_test': X_test_nat, 'y_test': y_test.values
}

# --- LUỒNG 3: RAW DATAFRAME (CHO DL) ---
train_df_dl = X_train_eng[original_num + cat_cols].copy()
test_df_dl = X_test_eng[original_num + cat_cols].copy()

for col in original_num:
    med = train_df_dl[col].median()
    train_df_dl[col] = train_df_dl[col].fillna(med)
    test_df_dl[col] = test_df_dl[col].fillna(med)
for col in cat_cols:
    mod = train_df_dl[col].mode()[0]
    train_df_dl[col] = train_df_dl[col].fillna(mod).astype(str)
    test_df_dl[col] = test_df_dl[col].fillna(mod).astype(str)

for i, c in enumerate(q_cols):
    train_df_dl[c] = Q_Train[:, i]
    test_df_dl[c] = Q_Test[:, i]

train_df_dl['loan_status'] = y_train.values
test_df_dl['loan_status'] = y_test.values

data_dict['dataframe'] = {
    'train': train_df_dl, 'test': test_df_dl,
    'num_cols': numeric_features, 'cat_cols': categorical_features
}

print("✅ Đã chuẩn bị xong 3 luồng SIÊU DUNG HỢP (QUANTUM_ONLY_ORIGINAL)!")


In [3]:
import time
import gc
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, roc_auc_score, f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings("ignore")

print("⚙️ KHỞI TẠO HỆ THỐNG LÕI, ĐÁNH GIÁ & DỌN DẸP VRAM...")

# 1. Khởi tạo danh sách lưu báo cáo toàn cục
business_reports = []
y_true_benchmark = data_dict['standard']['y_test']  # Nhãn test gốc để chấm điểm

# ==========================================
# KHO LƯU XÁC SUẤT - DÙNG CHO PHÂN TÍCH SAU
# ==========================================
# Dict này lưu xác suất dự đoán của MỌI model ngay khi train xong
# Mục đích: Sau khi del model (giải phóng RAM), vẫn giữ được y_probs để vẽ biểu đồ
all_probs = {}  # Cấu trúc: {'ModelName': np.array([0.1, 0.9, ...])}

# 2. Hàm Đánh Giá Đa Năng (Bao gồm Chẩn đoán Overfit/Underfit và Cắt Ngưỡng)
def evaluate_and_log(model_name, y_train_probs, y_train_true, y_test_probs):
    # --- PHẦN A: CHẨN ĐOÁN OVERFIT / UNDERFIT ---
    train_auc = roc_auc_score(y_train_true, y_train_probs)
    test_auc = roc_auc_score(y_true_benchmark, y_test_probs)
    gap = train_auc - test_auc
    
    print(f"\n📊 CHẨN ĐOÁN SỨC KHỎE [{model_name}]:")
    print(f" - Train AUC     : {train_auc:.4f}")
    print(f" - Test AUC      : {test_auc:.4f}")
    print(f" - Độ lệch (Gap) : {gap:.4f}")
    
    if train_auc < 0.75:
        print(" ⚠️ KẾT LUẬN: UNDERFITTING (Mô hình quá đơn giản, chưa nắm được quy luật nợ xấu).")
    elif gap > 0.05:
        print(" ⚠️ KẾT LUẬN: OVERFITTING (Mô hình học vẹt, thiếu tính tổng quát trên tập khách hàng mới).")
    else:
        print(" ✅ KẾT LUẬN: GOOD FIT! Mô hình cực kỳ ổn định và mạnh mẽ.")

        # --- PHẦN C: LƯU XÁC SUẤT VÀO KHO TOÀN CỤC ---
    all_probs[model_name] = y_test_probs.copy()
    print(f"✅ Đã lưu xác suất dự đoán cho {model_name} vào kho 'all_probs'!")

# Dọn rác khởi động
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


⚙️ KHỞI TẠO HỆ THỐNG LÕI, ĐÁNH GIÁ & DỌN DẸP VRAM...


## Cell 3A: Train XGBoost

In [4]:
import time
import xgboost as xgb
from sklearn.model_selection import train_test_split

name = 'XGBoost'
print(f"🚀 BẮT ĐẦU: {name} (SOTA Config)")
start_time = time.time()

# Data
X_tr_full, y_tr_full = data_dict['standard']['X_train'], data_dict['standard']['y_train']
X_te = data_dict['standard']['X_test']

# Trích xuất 20% dữ liệu làm bia ngắm cho Early Stopping
X_tr, X_val, y_tr, y_val = train_test_split(X_tr_full, y_tr_full, test_size=0.2, random_state=RANDOM_SEED)

# Cấu hình SOTA
model = xgb.XGBClassifier(
    random_state=RANDOM_SEED, 
    eval_metric='auc',
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=2.0,
    early_stopping_rounds=50
)

# Train & Predict
model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
y_train_probs = model.predict_proba(X_tr)[:, 1]
y_test_probs = model.predict_proba(X_te)[:, 1]

# 💾 LƯU MÔ HÌNH
model_path = f"saved_models/{name}_model.pkl"
joblib.dump(model, model_path)
print(f"💾 Đã lưu mô hình tại: {model_path}")

# Gọi hàm đánh giá
evaluate_and_log(name, y_train_probs, y_tr, y_test_probs)
print(f"⏱️ Thời gian: {time.time() - start_time:.1f} giây")

del model, X_tr, y_tr, X_val, y_val
gc.collect()

🚀 BẮT ĐẦU: XGBoost (SOTA Config)
💾 Đã lưu mô hình tại: saved_models/XGBoost_model.pkl

📊 CHẨN ĐOÁN SỨC KHỎE [XGBoost]:
 - Train AUC     : 0.9938
 - Test AUC      : 0.9437
 - Độ lệch (Gap) : 0.0502
 ⚠️ KẾT LUẬN: OVERFITTING (Mô hình học vẹt, thiếu tính tổng quát trên tập khách hàng mới).
✅ Đã lưu kết quả kinh doanh và xác suất cho XGBoost!
⏱️ Thời gian: 5.3 giây


0

## Cell 3B: Train LightGBM

In [5]:
import lightgbm as lgb

name = 'LightGBM'
print(f"🚀 BẮT ĐẦU: {name}")
start_time = time.time()

X_tr, y_tr = data_dict['native']['X_train'], data_dict['native']['y_train']
X_te = data_dict['native']['X_test']

model = lgb.LGBMClassifier(random_state=RANDOM_SEED, verbose=-1, categorical_feature=list(range(len(numeric_features), len(numeric_features)+len(categorical_features))))
if not USE_SMOTE_FOR_TREES:
    model.set_params(is_unbalance=True)

model.fit(X_tr, y_tr)
y_train_probs = model.predict_proba(X_tr)[:, 1]
y_test_probs = model.predict_proba(X_te)[:, 1]

# 💾 LƯU MÔ HÌNH
model_path = f"saved_models/{name}_model.pkl"
joblib.dump(model, model_path)
print(f"💾 Đã lưu mô hình tại: {model_path}")

evaluate_and_log(name, y_train_probs, y_tr, y_test_probs)
print(f"⏱️ Thời gian: {time.time() - start_time:.1f} giây")

del model, X_tr, y_tr, X_te
gc.collect()

🚀 BẮT ĐẦU: LightGBM
💾 Đã lưu mô hình tại: saved_models/LightGBM_model.pkl

📊 CHẨN ĐOÁN SỨC KHỎE [LightGBM]:
 - Train AUC     : 0.9798
 - Test AUC      : 0.9464
 - Độ lệch (Gap) : 0.0334
 ✅ KẾT LUẬN: GOOD FIT! Mô hình cực kỳ ổn định và mạnh mẽ.
✅ Đã lưu kết quả kinh doanh và xác suất cho LightGBM!
⏱️ Thời gian: 0.5 giây


249

## Cell 3C: Train CatBoost

In [6]:
import time
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split

name = 'CatBoost'
print(f"🚀 BẮT ĐẦU: {name} (SOTA Config)")
start_time = time.time()

X_tr_full, y_tr_full = data_dict['native']['X_train'], data_dict['native']['y_train']
X_te = data_dict['native']['X_test']

cat_idx = list(range(len(numeric_features), len(numeric_features)+len(categorical_features)))
X_tr_cat_full = np.array(X_tr_full, dtype=object)
X_te_cat = np.array(X_te, dtype=object)
X_tr_cat_full[:, cat_idx] = X_tr_cat_full[:, cat_idx].astype(int).astype(str)
X_te_cat[:, cat_idx] = X_te_cat[:, cat_idx].astype(int).astype(str)

X_tr, X_val, y_tr, y_val = train_test_split(X_tr_cat_full, y_tr_full, test_size=0.2, random_state=RANDOM_SEED)

model = CatBoostClassifier(
    random_state=RANDOM_SEED, 
    verbose=0, 
    cat_features=cat_idx,
    iterations=1000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5.0,
    early_stopping_rounds=50,
    eval_metric='AUC'
)

if not USE_SMOTE_FOR_TREES:
    model.set_params(auto_class_weights='Balanced')

model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)])
y_train_probs = model.predict_proba(X_tr)[:, 1]
y_test_probs = model.predict_proba(X_te_cat)[:, 1]

# 💾 LƯU MÔ HÌNH
model_path = f"saved_models/{name}_model.cbm"
model.save_model(model_path)
print(f"💾 Đã lưu CatBoost tại: {model_path}")

evaluate_and_log(name, y_train_probs, y_tr, y_test_probs)
print(f"⏱️ Thời gian: {time.time() - start_time:.1f} giây")

del model, X_tr, y_tr, X_val, y_val, X_te_cat, X_tr_cat_full
gc.collect()

🚀 BẮT ĐẦU: CatBoost (SOTA Config)
💾 Đã lưu CatBoost tại: saved_models/CatBoost_model.cbm

📊 CHẨN ĐOÁN SỨC KHỎE [CatBoost]:
 - Train AUC     : 0.9782
 - Test AUC      : 0.9447
 - Độ lệch (Gap) : 0.0335
 ✅ KẾT LUẬN: GOOD FIT! Mô hình cực kỳ ổn định và mạnh mẽ.
✅ Đã lưu kết quả kinh doanh và xác suất cho CatBoost!
⏱️ Thời gian: 18.1 giây


0

## Cell 3D: Train NGBoost

In [7]:
import time
import joblib
import numpy as np
from ngboost import NGBClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split

name = 'NGBoost'
print(f"🚀 BẮT ĐẦU: {name} (SOTA Config)")
start_time = time.time()

X_tr_full, y_tr_full = data_dict['standard']['X_train'], data_dict['standard']['y_train']
X_te = data_dict['standard']['X_test']

X_tr, X_val, y_tr, y_val = train_test_split(X_tr_full, y_tr_full, test_size=0.2, random_state=RANDOM_SEED)

base_learner = DecisionTreeRegressor(criterion='friedman_mse', max_depth=4, random_state=RANDOM_SEED)

model = NGBClassifier(
    Base=base_learner,
    random_state=RANDOM_SEED, 
    n_estimators=1000,
    learning_rate=0.02,
    minibatch_frac=0.8,
    col_sample=0.8,
    verbose=True, 
    verbose_eval=50
)

model.fit(
    X_tr, y_tr, 
    X_val=X_val, Y_val=y_val, 
    early_stopping_rounds=50
)

y_train_probs = model.predict_proba(X_tr)[:, 1]
y_test_probs = model.predict_proba(X_te)[:, 1]

evaluate_and_log(name, y_train_probs, y_tr, y_test_probs)

# 💾 LƯU MÔ HÌNH
model_path = f"saved_models/{name}_model.pkl"
joblib.dump(model, model_path)
print(f"💾 Đã lưu NGBoost tại: {model_path}")

print(f"⏱️ Thời gian: {time.time() - start_time:.1f} giây")

del model, X_tr, y_tr, X_val, y_val, X_te, X_tr_full, y_tr_full
gc.collect()

🚀 BẮT ĐẦU: NGBoost (SOTA Config)
[iter 0] loss=0.6931 val_loss=0.6740 scale=2.0000 norm=4.0000
[iter 50] loss=0.3629 val_loss=0.3690 scale=2.0000 norm=3.1210
[iter 100] loss=0.3092 val_loss=0.3171 scale=1.0000 norm=1.5340
[iter 150] loss=0.2940 val_loss=0.3015 scale=1.0000 norm=1.5355
[iter 200] loss=0.2852 val_loss=0.2935 scale=2.0000 norm=3.0600
[iter 250] loss=0.2802 val_loss=0.2892 scale=1.0000 norm=1.5291
[iter 300] loss=0.2763 val_loss=0.2859 scale=0.5000 norm=0.7669
[iter 350] loss=0.2723 val_loss=0.2817 scale=0.5000 norm=0.7648
[iter 400] loss=0.2697 val_loss=0.2803 scale=1.0000 norm=1.5312
[iter 450] loss=0.2704 val_loss=0.2791 scale=0.5000 norm=0.7679
[iter 500] loss=0.2675 val_loss=0.2780 scale=0.0156 norm=0.0239
[iter 550] loss=0.2683 val_loss=0.2767 scale=0.0156 norm=0.0240
[iter 600] loss=0.2656 val_loss=0.2762 scale=0.5000 norm=0.7666
[iter 650] loss=0.2662 val_loss=0.2753 scale=1.0000 norm=1.5383
[iter 700] loss=0.2614 val_loss=0.2745 scale=0.2500 norm=0.3814
[iter 750]

71

## Cell 3E: Train TabNet

In [8]:
import time
import gc
import numpy as np
import torch
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.model_selection import train_test_split

name = 'TabNet'
print(f"🚀 BẮT ĐẦU: {name} (SOTA Config)")
start_time = time.time()

X_tr_full = data_dict['standard']['X_train'].astype(np.float32)
y_tr_full = data_dict['standard']['y_train'].astype(np.int64)
X_te = data_dict['standard']['X_test'].astype(np.float32)

X_tr, X_val, y_tr, y_val = train_test_split(X_tr_full, y_tr_full, test_size=0.2, random_state=RANDOM_SEED)

model = TabNetClassifier(
    n_d=16, n_a=16,
    n_steps=4,
    gamma=1.3,
    lambda_sparse=1e-3,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2, weight_decay=1e-5),
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    scheduler_params={"step_size": 20, "gamma": 0.5},
    mask_type="entmax",
    seed=RANDOM_SEED, 
    verbose=1
)

model.fit(
    X_train=X_tr, y_train=y_tr,
    eval_set=[(X_val, y_val)],
    eval_name=['valid'],
    eval_metric=['auc'],
    max_epochs=200,
    patience=20,
    batch_size=512,
    virtual_batch_size=128
)

y_train_probs = model.predict_proba(X_tr)[:, 1]
y_test_probs = model.predict_proba(X_te)[:, 1]

evaluate_and_log(name, y_train_probs, y_tr, y_test_probs)
print(f"⏱️ Thời gian: {time.time() - start_time:.1f} giây")

# 💾 LƯU MÔ HÌNH
model_path = f"saved_models/{name}_model"
saved_filepath = model.save_model(model_path)
print(f"💾 Đã lưu TabNet tại: {saved_filepath}")

del model, X_tr, y_tr, X_val, y_val, X_te, X_tr_full, y_tr_full
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

🚀 BẮT ĐẦU: TabNet (SOTA Config)
epoch 0  | loss: 0.51556 | valid_auc: 0.87266 |  0:00:03s
epoch 1  | loss: 0.40105 | valid_auc: 0.89308 |  0:00:05s
epoch 2  | loss: 0.37356 | valid_auc: 0.91178 |  0:00:07s
epoch 3  | loss: 0.3506  | valid_auc: 0.91935 |  0:00:10s
epoch 4  | loss: 0.34222 | valid_auc: 0.91528 |  0:00:12s
epoch 5  | loss: 0.33432 | valid_auc: 0.91945 |  0:00:15s
epoch 6  | loss: 0.32922 | valid_auc: 0.91952 |  0:00:17s
epoch 7  | loss: 0.32434 | valid_auc: 0.92324 |  0:00:19s
epoch 8  | loss: 0.32127 | valid_auc: 0.92697 |  0:00:22s
epoch 9  | loss: 0.32245 | valid_auc: 0.9224  |  0:00:24s
epoch 10 | loss: 0.3186  | valid_auc: 0.93224 |  0:00:27s
epoch 11 | loss: 0.31067 | valid_auc: 0.93027 |  0:00:29s
epoch 12 | loss: 0.3118  | valid_auc: 0.9299  |  0:00:31s
epoch 13 | loss: 0.30653 | valid_auc: 0.92833 |  0:00:34s
epoch 14 | loss: 0.30524 | valid_auc: 0.92582 |  0:00:36s
epoch 15 | loss: 0.29964 | valid_auc: 0.93713 |  0:00:38s
epoch 16 | loss: 0.30161 | valid_auc: 0.

## Cell 3F: Train FT-Transformer

In [9]:
from pytorch_tabular import TabularModel
from pytorch_tabular.models import FTTransformerConfig
from pytorch_tabular.config import DataConfig, TrainerConfig, OptimizerConfig

name = 'FT-Transformer'
print(f"🚀 BẮT ĐẦU: {name} (DL Hạng Nặng)")
start_time = time.time()

train_df = data_dict['dataframe']['train']
test_df = data_dict['dataframe']['test']

data_cfg = DataConfig(target=['loan_status'], continuous_cols=numeric_features, categorical_cols=categorical_features)

trainer_cfg = TrainerConfig(
    batch_size=256,
    max_epochs=200,
    early_stopping_patience=20,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    progress_bar="none"
)

opt_cfg = OptimizerConfig(
    optimizer="AdamW",
    optimizer_params={"weight_decay": 1e-4},
    lr_scheduler="ReduceLROnPlateau",
    lr_scheduler_params={"patience": 5, "factor": 0.5}
)

ft_config = FTTransformerConfig(
    task="classification",
    learning_rate=1e-3,
    num_attn_blocks=6,
    input_embed_dim=64,
    num_heads=8,
    attn_dropout=0.2,
    ff_dropout=0.2
)

model = TabularModel(data_config=data_cfg, model_config=ft_config, optimizer_config=opt_cfg, trainer_config=trainer_cfg)
model.fit(train=train_df)

preds_train_df = model.predict(train_df, ret_logits=False)
preds_test_df = model.predict(test_df, ret_logits=False)

prob_cols = [col for col in preds_test_df.columns if 'probability' in col.lower()]
prob_col = sorted(prob_cols)[-1]

y_train_probs = preds_train_df[prob_col].values
y_test_probs = preds_test_df[prob_col].values
y_tr_true = train_df['loan_status'].values

evaluate_and_log(name, y_train_probs, y_tr_true, y_test_probs)
print(f"⏱️ Thời gian: {time.time() - start_time:.1f} giây")

# 💾 LƯU MÔ HÌNH
model_dir = f"saved_models/{name}_model"
model.save_model(model_dir)
print(f"💾 Đã lưu FT-Transformer tại: {model_dir}")

del model, train_df, test_df, preds_train_df, preds_test_df
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

2026-07-28 16:59:28,196 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
Seed set to 42
2026-07-28 16:59:28,216 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-07-28 16:59:28,253 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
2026-07-28 16:59:28,347 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: FTTransformerModel


🚀 BẮT ĐẦU: FT-Transformer (DL Hạng Nặng)


2026-07-28 16:59:28,417 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-07-28 16:59:28,791 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ FTTransformerBackbone │  1.1 M │ train │     0 │
│ 1 │ _embedding_layer │ Embedding2dLayer      │  3.7 K │ train │     0 │
│ 2 │ _head            │ LinearHead            │    130 │ train │     0 │
│ 3 │ loss             │ CrossEntropyLoss      │      0 │ train │     0 │
└───┴──────────────────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 1.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.1 M                                                                                                
Total estimated model params size (MB): 4.349                                                                      
Modules in train mode: 130                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

2026-07-28 17:02:31,268 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-07-28 17:02:31,269 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model
`weights_only` was not set, defaulting to `False`.



📊 CHẨN ĐOÁN SỨC KHỎE [FT-Transformer]:
 - Train AUC     : 0.9353
 - Test AUC      : 0.9282
 - Độ lệch (Gap) : 0.0072
 ✅ KẾT LUẬN: GOOD FIT! Mô hình cực kỳ ổn định và mạnh mẽ.
✅ Đã lưu kết quả kinh doanh và xác suất cho FT-Transformer!
⏱️ Thời gian: 203.4 giây
💾 Đã lưu FT-Transformer tại: saved_models/FT-Transformer_model


## Cell 3G: Train NODE

In [10]:
from pytorch_tabular import TabularModel
from pytorch_tabular.models import NodeConfig
from pytorch_tabular.config import DataConfig, TrainerConfig, OptimizerConfig

name = 'NODE'
print(f"🚀 BẮT ĐẦU: {name} (DL Hạng Nặng)")
start_time = time.time()

train_df = data_dict['dataframe']['train']
test_df = data_dict['dataframe']['test']

data_cfg = DataConfig(target=['loan_status'], continuous_cols=numeric_features, categorical_cols=categorical_features)

trainer_cfg = TrainerConfig(
    batch_size=256,
    max_epochs=200,
    early_stopping_patience=20,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    progress_bar="none"
)

opt_cfg = OptimizerConfig(
    optimizer="AdamW",
    optimizer_params={"weight_decay": 1e-4},
    lr_scheduler="ReduceLROnPlateau",
    lr_scheduler_params={"patience": 5, "factor": 0.5}
)

node_config = NodeConfig(
    task="classification",
    learning_rate=1e-3, 
    num_layers=4,
    num_trees=1024,
    depth=6
)

model = TabularModel(data_config=data_cfg, model_config=node_config, optimizer_config=opt_cfg, trainer_config=trainer_cfg)
model.fit(train=train_df)

preds_train_df = model.predict(train_df, ret_logits=False)
preds_test_df = model.predict(test_df, ret_logits=False)

prob_cols = [col for col in preds_test_df.columns if 'probability' in col.lower()]
prob_col = sorted(prob_cols)[-1]

y_train_probs = preds_train_df[prob_col].values
y_test_probs = preds_test_df[prob_col].values
y_tr_true = train_df['loan_status'].values

evaluate_and_log(name, y_train_probs, y_tr_true, y_test_probs)
print(f"⏱️ Thời gian: {time.time() - start_time:.1f} giây")

# 💾 LƯU MÔ HÌNH
model_dir = f"saved_models/{name}_model"
model.save_model(model_dir)
print(f"💾 Đã lưu NODE tại: {model_dir}")

del model, train_df, test_df, preds_train_df, preds_test_df
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

2026-07-28 17:02:52,337 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
Seed set to 42
2026-07-28 17:02:52,353 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-07-28 17:02:52,379 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
2026-07-28 17:02:52,464 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: NODEModel


🚀 BẮT ĐẦU: NODE (DL Hạng Nặng)


2026-07-28 17:02:53,624 - {pytorch_tabular.models.node.node_model:74} - INFO - Data Aware Initialization of NODE using a forward pass with 2000 batch size....
2026-07-28 17:04:27,318 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-07-28 17:04:27,369 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ NODEBackbone     │  190 M │ train │     0 │
│ 1 │ _embedding_layer │ Embedding1dLayer │    111 │ train │     0 │
│ 2 │ _head            │ Lambda           │      0 │ train │     0 │
│ 3 │ loss             │ CrossEntropyLoss │      0 │ train │     0 │
└───┴──────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 190 M                                                                                            
Non-trainable params: 3.1 K                                                                                        
Total params: 190 M                                                                                                
Total estimated model params size (MB): 763.179                                                                    
Modules in train mode: 16                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

2026-07-28 18:24:24,836 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-07-28 18:24:24,838 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model



📊 CHẨN ĐOÁN SỨC KHỎE [NODE]:
 - Train AUC     : 0.9384
 - Test AUC      : 0.9173
 - Độ lệch (Gap) : 0.0211
 ✅ KẾT LUẬN: GOOD FIT! Mô hình cực kỳ ổn định và mạnh mẽ.
✅ Đã lưu kết quả kinh doanh và xác suất cho NODE!
⏱️ Thời gian: 9957.8 giây


`weights_only` was not set, defaulting to `False`.


💾 Đã lưu NODE tại: saved_models/NODE_model


## BẢNG ĐIỀU KHIỂN CẮT NGƯỠNG TỰ ĐỘNG (THRESHOLD CONFIGURATION)
Chỉ cần chạy Cell này để tính lại các mốc kinh doanh mà **KHÔNG CẦN TRAIN LẠI MODEL**.

In [ ]:
# ==========================================
# ⚙️ BẢNG ĐIỀU KHIỂN THRESHOLD (THRESHOLD DASHBOARD)
# ==========================================
TARGET_PRECISION = 0.98   # Ngưỡng An toàn mong muốn (Tự do thay đổi)
TARGET_RECALL = 0.95      # Ngưỡng Quét sạch mong muốn (Tự do thay đổi)

business_reports = []

for model_name, p_arr in all_probs.items():
    precisions, recalls, thresholds = precision_recall_curve(y_true_benchmark, p_arr)
    f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-10)
    
    # Max F1
    t_f1 = thresholds[np.argmax(f1_scores)]
    
    # Theo Target Precision
    idx_p = np.where(precisions[:-1] >= TARGET_PRECISION)[0]
    t_prec = thresholds[idx_p[0]] if len(idx_p) > 0 else thresholds[np.argmax(precisions[:-1])]
    
    # Theo Target Recall
    idx_r = np.where(recalls[:-1] >= TARGET_RECALL)[0]
    t_rec = thresholds[idx_r[-1]] if len(idx_r) > 0 else thresholds[np.argmax(recalls[:-1])]
    
    choices = [
        ('Cân Bằng (Max F1)', t_f1),
        (f'An Toàn (Prec ≥ {TARGET_PRECISION})', t_prec),
        (f'Quét Sạch (Rec ≥ {TARGET_RECALL})', t_rec)
    ]
    
    for ver_name, t in choices:
        y_pred = (p_arr >= t).astype(int)
        business_reports.append({
            'Model Core': model_name,
            'Phiên bản Kinh doanh': ver_name,
            'Threshold Mở Khóa': t,
            'ROC AUC': roc_auc_score(y_true_benchmark, p_arr),
            'F1 Score': f1_score(y_true_benchmark, y_pred),
            'Precision': precision_score(y_true_benchmark, y_pred, zero_division=0),
            'Recall': recall_score(y_true_benchmark, y_pred, zero_division=0)
        })

# Xuất Báo Cáo
report_df = pd.DataFrame(business_reports).set_index(['Model Core', 'Phiên bản Kinh doanh'])
print("🏆 BẢNG TỔNG HỢP MA TRẬN KINH DOANH CHO CÁC MÔ HÌNH SOTA:")
display(report_df.round(4))
report_df.to_excel('SOTA_Business_Threshold_Report.xlsx')
print("\n✅ Đã lưu báo cáo ra file: SOTA_Business_Threshold_Report.xlsx")
